# Megtévesztő szállodai vélemények felismerése magyar nyelven
**Molnár Barnabás (B4R87N)**
Mesterséges intelligencia alkalmazási területei – NBXMA1HMLF/MA1_EA_AT, féléves feladat

Kapcsolódik a diplomamunkámhoz: *Megtévesztő vélemények detektálása magyar nyelven transformer-alapú hibrid CNN–BiLSTM modellel*

**Adat:** Ott-féle Deceptive Opinion Spam Corpus (1600 angol szállodai vélemény, 20 szálloda), gépi fordítással magyarra.

**Modellek:**
1. TF-IDF + logisztikus regresszió (baseline)
2. huBERT (`SZTAKI-HLT/hubert-base-cc`) finomhangolás
3. huBERT + CNN–BiLSTM hibrid

**Kiértékelés:** 5-szörös keresztvalidáció, szállodák szerint csoportosítva (egy szálloda minden véleménye ugyanabba a foldba kerül); pontosság, makro-F1, McNemar-próba.

In [ ]:
REPO_URL = "https://github.com/brnbs15/hu-deceptive-reviews.git"

import os

!nvidia-smi -L

if not os.path.exists("/content/hu-deceptive-reviews"):
    !git clone -q {REPO_URL} /content/hu-deceptive-reviews

%cd /content/hu-deceptive-reviews
!pip install -q -r requirements.txt

## 1. Gépi fordítás
A véleményeket mondatokra bontva fordítjuk le a `Helsinki-NLP/opus-mt-en-hu` modellel (beam = 4).

In [ ]:
!python -m src.translate --engine opus --batch-size 64 --sample-size 32

In [ ]:
# pár példa a fordításra
import pandas as pd
pd.set_option("display.max_colwidth", 400)

hu = pd.read_csv("data/processed/reviews_hu.csv")
hu[["deceptive", "polarity", "text_en", "text_hu"]].sample(3, random_state=1)

## 2. A fordítás kézi ellenőrzése
32 véleményes minta (a 4 csoportból – valódi/megtévesztő × pozitív/negatív – 8-8), amin kézzel értékeljük a fordítást:
* `adequacy_1_5` – hűség az eredetihez (1–5)
* `fluency_1_5` – gördülékenység (1–5)
* `cue_preserved_i_n` – megmaradtak-e a megtévesztésre utaló jelek (i/n)
* `notes` – megjegyzés

In [ ]:
from google.colab import files
files.download("data/processed/mt_quality_sample.csv")

In [ ]:
# kitöltött értékelés visszatöltése
UPLOAD_RATINGS = False

if UPLOAD_RATINGS:
    from google.colab import files
    uploaded = files.upload()
    for name, content in uploaded.items():
        with open("data/processed/mt_quality_sample.csv", "wb") as f:
            f.write(content)
        print("Mentve:", name)

## 3. Baseline: TF-IDF + logisztikus regresszió
A magyar fordításon, és összehasonlításként az angol eredetin.

In [ ]:
!python -m src.baseline --lang hu
!python -m src.baseline --lang en

## 4. huBERT finomhangolás
4 epoch, batch 16, lr = 2·10⁻⁵, max. 512 token, fp16.

In [ ]:
!python -m src.neural --model hubert

## 5. Hibrid modell: huBERT + CNN–BiLSTM
huBERT token-kimenetek → Conv1D (k = 3, 5; 2×128 szűrő) → BiLSTM (2×128) → max + átlag pooling → lineáris réteg.
Tanulási ráta: huBERT 2·10⁻⁵, a többi réteg 10⁻³.

In [ ]:
!python -m src.neural --model hybrid

## 6. Eredmények

In [ ]:
!python -m src.analyze

from IPython.display import Image, display
display(Image("results/fig_results.png", width=800))
display(Image("results/fig_confusion.png", width=800))

## 7. Eredmények mentése

In [ ]:
!cd /content && rm -f eredmenyek.zip && zip -q -r eredmenyek.zip hu-deceptive-reviews/results hu-deceptive-reviews/data/processed
from google.colab import files
files.download("/content/eredmenyek.zip")